## Importing Libraries

In [14]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import nltk 
nltk.download('wordnet')
nltk.download('punkt_tab')
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\senas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\senas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Loading Dataset

In [15]:
df = pd.read_csv('./../data/processed/20newsgroup_preprocessed_own.csv', on_bad_lines='skip', delimiter=";")
print(df.shape)
df.head()

(18828, 3)


,target,text,text_cleaned
0,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,alt.atheism,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: R...,kings become philosophers philosophers become ...
4,alt.atheism,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


In [16]:
df = df.dropna(subset=['text_cleaned'])
print(df.shape)

(18792, 3)


## Lemmatization

In [6]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = nltk.word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmas)

df['text_cleaned'] = df['text_cleaned'].apply(lambda x: lemmatize_text(x))
df.head()

,target,text,text_cleaned
0,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,alt.atheism,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: R...,king become philosopher philosopher become kin...
4,alt.atheism,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


## Encoding labels

In [17]:
# Create and fit the encoder on the original target column
label_encoder = LabelEncoder()
df['target'] = label_encoder.fit_transform(df['target'])

# Print to verify
print("Unique labels in df (encoded):", np.unique(df['target']))
print("Label classes:", label_encoder.classes_)
df.head()

Unique labels in df (encoded): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
Label classes: ['alt.atheism' 'comp.graphics' 'comp.os.ms-windows.misc'
 'comp.sys.ibm.pc.hardware' 'comp.sys.mac.hardware' 'comp.windows.x'
 'misc.forsale' 'rec.autos' 'rec.motorcycles' 'rec.sport.baseball'
 'rec.sport.hockey' 'sci.crypt' 'sci.electronics' 'sci.med' 'sci.space'
 'soc.religion.christian' 'talk.politics.guns' 'talk.politics.mideast'
 'talk.politics.misc' 'talk.religion.misc']


,target,text,text_cleaned
0,0,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,0,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,0,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,0,From: mathew <mathew@mantis.co.uk>\nSubject: R...,kings become philosophers philosophers become ...
4,0,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


## Split the dataset

In [18]:
X_train, X_test, y_train, y_test = train_test_split(df['text_cleaned'], df['target'], test_size=0.2, stratify=df['target'], random_state=42)

## Chunk the documents

In [19]:
def chunk_text(text, chunk_size=50, overlap=15):
    words = text.split()
    if len(words) < chunk_size:
        return [' '.join(words)]  # Keep short texts as a single chunk
    return [' '.join(words[i:i+chunk_size])
            for i in range(0, len(words) - chunk_size + 1, chunk_size - overlap)]

# New lists to store chunked data
X_train_chunked = []
y_train_chunked = []

# Apply chunking
for text, label in zip(X_train, y_train):
    chunks = chunk_text(text, chunk_size=150, overlap=25)
    X_train_chunked.extend(chunks)
    y_train_chunked.extend([label] * len(chunks))

print(f"Original X_train size: {len(X_train)}")
print(f"Chunked X_train size: {len(X_train_chunked)}")

Original X_train size: 15033
Chunked X_train size: 19364


## Tokenizing the documents

In [20]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=50000, stop_words="english", ngram_range = (1,3))

# Transform text data into TF-IDF features
X_tfidf = vectorizer.fit_transform(X_train_chunked)
selector = SelectKBest(chi2, k=5000)
X_selected = selector.fit_transform(X_tfidf, y_train_chunked)
X_selected = X_selected.toarray()

# Show shape of transformed data
print("TF-IDF Matrix Shape:", X_selected.shape)

TF-IDF Matrix Shape: (19364, 5000)


Preview the vectorized documents

In [24]:
print(selector.get_feature_names_out()[:20])
print(vectorizer.get_feature_names_out()[:20])

['x2' 'x3' 'x15' 'x18' 'x19' 'x93' 'x107' 'x111' 'x117' 'x127' 'x157'
 'x187' 'x188' 'x194' 'x195' 'x203' 'x221' 'x261' 'x262' 'x270']
['aab' 'aachende' 'aamir' 'aamir qazi' 'aamir qazi care' 'aan' 'aap'
 'aaron' 'aaron bryce' 'aaron bryce cardenas' 'aaron ray'
 'aaron ray clements' 'abandon' 'abandoned' 'abandoning' 'abate' 'abbott'
 'abbreviation' 'abc' 'abc coverage']


## Tokenize test data

In [25]:
# Vectorize the test data using the same vectorizer
X_test_tfidf = vectorizer.transform(X_test)

# Select the same features using the same selector
X_test_selected = selector.transform(X_test_tfidf)

# Optionally convert to array if your model expects that
X_test_selected = X_test_selected.toarray()

## Training the model

In [28]:
# Initialize and train the model
nb_classifier = MultinomialNB()
nb_classifier.fit(X_tfidf, y_train_chunked)

MultinomialNB()

## Evaluate the model

In [29]:
# Predict on test set
y_pred = nb_classifier.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
print("Model Accuracy:", accuracy)
print("Model Precision:", precision)
print("Model Recall:", recall)
print("Model F1 Score:", f1)

Model Accuracy: 0.8292098962490024
Model Precision: 0.8530300722597414
Model Recall: 0.8182310891248337
Model F1 Score: 0.8199906149018069


In [32]:
new_texts = ["This means that any University desktop or laptop PCs which are still using Windows 10 will need to be upgraded to Windows 11 or replaced with more up-to-date equipment if they are unable to support Windows 11, as otherwise they will leave the University exposed to an increased cyber risk. This work is mandatory; any PCs which are still running Windows 10 will be blocked from the University network at some point in the future. Soon, colleagues and postgraduate researchers with Windows 10 PCs will be able to arrange their own upgrade, and choose a suitable time slot for this.", 
             "This week, a busy launch manifest saw SpaceX launch the Crew-10 and two Starlink missions from Florida and the delayed SPHEREx/PUNCH and Transporter 13 missions from California.Outside of the United States, Rocket Lab flew an Electron from New Zealand. A Chang Zhang 8 rocket launched from a new commercial pad in China which also launched two other rockets. A Russian Angara rocket launched an unknown payload from Plesetsk."]
lemmatized_new_texts = []
for text in new_texts:
    lemmatized_new_texts.append(lemmatize_text(text)) 
print(lemmatized_new_texts)
new_texts_tfidf = vectorizer.transform(lemmatized_new_texts)  # Transform using the trained vectorizer

print(label_encoder.classes_)

['This mean that any University desktop or laptop PCs which are still using Windows 10 will need to be upgraded to Windows 11 or replaced with more up-to-date equipment if they are unable to support Windows 11 , a otherwise they will leave the University exposed to an increased cyber risk . This work is mandatory ; any PCs which are still running Windows 10 will be blocked from the University network at some point in the future . Soon , colleague and postgraduate researcher with Windows 10 PCs will be able to arrange their own upgrade , and choose a suitable time slot for this .', 'This week , a busy launch manifest saw SpaceX launch the Crew-10 and two Starlink mission from Florida and the delayed SPHEREx/PUNCH and Transporter 13 mission from California.Outside of the United States , Rocket Lab flew an Electron from New Zealand . A Chang Zhang 8 rocket launched from a new commercial pad in China which also launched two other rocket . A Russian Angara rocket launched an unknown payload

In [33]:
predictions = nb_classifier.predict(new_texts_tfidf)
print(label_encoder.classes_[predictions])

['comp.os.ms-windows.misc' 'sci.space']


## BERT experiment

In [ ]:
# Load pre-trained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")